In [ ]:
# setup and load
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

DATE_COLS = ['Invoice Date', 'Customer First Invoice Date']
df = pd.read_csv('../data/processed/data_clean_final.csv', low_memory=False, parse_dates=DATE_COLS)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Line revenue
df['line_revenue'] = df['Invoiced price'] * df['Invoiced qty (shipped)']
# ne znam je li ovo okej ili  je u invoiced price uracunato qty

# Referentni datum
snapshot_date = df['Invoice Date'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date}')

# RFM agregacija
rfm = df.groupby('CustomerID').agg(
    Recency=('Invoice Date', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice #', 'nunique'),
    Monetary=('line_revenue', 'sum')
).reset_index()

print(f'Kupaca: {len(rfm)}')
print(rfm.head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    np.log1p(rfm[col]).hist(bins=50, ax=ax, edgecolor='black')
    ax.set_title(f'log1p({col})')
    ax.set_ylabel('Broj kupaca')
plt.tight_layout()
plt.show()

# Osnovne statistike
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

In [ ]:
# Top 10 kupaca po svakoj dimenziji
print("=== Top 10 Frequency ===")
print(rfm.nlargest(10, 'Frequency')[['CustomerID', 'Recency', 'Frequency', 'Monetary']])

print("\n=== Top 10 Monetary ===")
print(rfm.nlargest(10, 'Monetary')[['CustomerID', 'Recency', 'Frequency', 'Monetary']])

print("\n=== Osnovne statistike ===")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe(percentiles=[.25, .5, .75, .9, .95, .99]))

In [ ]:
# Recency: niži = bolji → invertirana skala (5 = najnoviji)
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])
# Frequency i Monetary: viši = bolji
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_score'] = pd.qcut(rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5])

rfm['RFM_score'] = rfm['R_score'].astype(int) + rfm['F_score'].astype(int) + rfm['M_score'].astype(int)

print(rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_score', 'F_score', 'M_score', 'RFM_score']].head(10))

In [ ]:
# Concatenated segment string
rfm['RFM_segment'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

# Humane labele bazirane na R i F scoreovima (standard mapping)
seg_map = {
    r'5[4-5]': 'Champions',
    r'[3-4][4-5]': 'Loyal Customers',
    r'[4-5][2-3]': 'Potential Loyalists',
    r'51': 'New Customers',
    r'41': 'Promising',
    r'3[1-2]' : 'About To Sleep',
    r'33': 'Need Attention',
    r'[1-2][3-4]': 'At Risk',
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2]5': 'Cant Lose Them',
}

def assign_segment(row):
    rf = str(row['R_score']) + str(row['F_score'])
    for pattern, label in seg_map.items():
        import re
        if re.match(pattern, rf):
            return label
    return 'Other'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)
print(rfm['Segment'].value_counts())

In [ ]:
# Log1p da izbjegnemo log(0)
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].copy()
rfm_log['Monetary'] = np.log1p(rfm_log['Monetary'])
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Recency'] = np.log1p(rfm_log['Recency'])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

# Provjera distribucija nakon transformacije
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, rfm_scaled.columns):
    rfm_scaled[col].hist(bins=50, ax=ax, edgecolor='black')
    ax.set_title(f'{col} (log + scaled)')
plt.tight_layout()
plt.show()

In [ ]:
inertia = []
sil_scores = []
K_range = range(2, 8)

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(rfm_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertia, 'bo-')
ax1.set_xlabel('K')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')

ax2.plot(K_range, sil_scores, 'ro-')
ax2.set_xlabel('K')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Scores')

plt.tight_layout()
plt.show()

for k, s in zip(K_range, sil_scores):
    print(f'K={k}: silhouette={s:.4f}')

# cini se da bi K=4 bio nekako najoptimalniji

In [ ]:
for k in [3, 4]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    rfm[f'K{k}'] = km.fit_predict(rfm_scaled)
    print(f'\n=== K={k} ===')
    print(rfm.groupby(f'K{k}')[['Recency', 'Frequency', 'Monetary']].mean().round(1))
    print(f'Veličine: {rfm[f"K{k}"].value_counts().sort_index().to_dict()}')

In [ ]:
OPTIMAL_K = 4

km_final = KMeans(n_clusters=OPTIMAL_K, n_init=10, random_state=42)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled)

# Profiliranje klastera
cluster_profile = rfm.groupby('Cluster').agg(
    Recency_mean=('Recency', 'mean'),
    Frequency_mean=('Frequency', 'mean'),
    Monetary_mean=('Monetary', 'mean'),
    Monetary_median=('Monetary', 'median'),
    Count=('CustomerID', 'count')
).round(1)

print(cluster_profile)

In [ ]:
rfm_scaled['Cluster'] = rfm['Cluster'].values
cluster_means = rfm_scaled.groupby('Cluster').mean()

fig, ax = plt.subplots(figsize=(10, 6))
for cluster in cluster_means.index:
    ax.plot(['Recency', 'Frequency', 'Monetary'],
            cluster_means.loc[cluster],
            marker='o', label=f'Cluster {cluster}')
ax.set_ylabel('Scaled Mean')
ax.set_title('Customer Segment Profiles (Snake Plot)')
ax.legend()
ax.axhline(0, color='grey', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    rfm,
    x=rfm_scaled['Recency'],
    y=rfm_scaled['Frequency'],
    z=rfm_scaled['Monetary'],
    color=rfm['Cluster'].astype(str),
    opacity=0.4,
    title='K-Means Clusters (3D)',
    labels={'x': 'Recency', 'y': 'Frequency', 'z': 'Monetary', 'color': 'Cluster'}
)
fig.write_html('../visualization/clusters_3d.html')
print('Otvori clusters_3d.html u browseru!')